In [6]:
import pandas as pd 
import numpy as np 
from pathlib import Path
import argparse
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
data = pd.read_csv("/var/www/Receipt-Intelligence/data/combined.csv")
data.head()

,text,category
0,OANDO petrol,transport
1,MEDPLUS prescription bandage,health
2,NETFLIX amusement park ticket concert ticket v...,entertainment
3,ALPHA PHARMACY face mask hand sanitizer,health
4,CVS PHARMACY antibiotics first aid,health


In [29]:
train, test = train_test_split(data, test_size=0.2, random_state=42, stratify=data['category'])

In [30]:
X_train = train['text']
y_train = train['category']
X_test = test['text']   
y_test = test['category']

In [7]:
MODELS: dict[str, tuple[Pipeline, dict]] = {
    "LogisticRegression": (
        Pipeline([
            ("tfidf", TfidfVectorizer(ngram_range=(1, 2))),
            ("clf", LogisticRegression(max_iter=1000)),
        ]),
        {"clf__C": [0.1, 1, 10], "clf__class_weight": [None, "balanced"]},
    ),
    "MultinomialNB": (
        Pipeline([
            ("tfidf", TfidfVectorizer(ngram_range=(1, 2))),
            ("clf", MultinomialNB()),
        ]),
        {"clf__alpha": [0.1, 0.5, 1.0]},
    ),
    "LinearSVC": (
        Pipeline([
            ("tfidf", TfidfVectorizer(ngram_range=(1, 2))),
            ("clf", LinearSVC(max_iter=5000)),
        ]),
        {"clf__C": [0.1, 1, 10], "clf__class_weight": [None, "balanced"]},
    ),
    "RandomForest": (
        Pipeline([
            ("tfidf", TfidfVectorizer(ngram_range=(1, 2))),
            ("clf", RandomForestClassifier(random_state=42)),
        ]),
        {
            "clf__n_estimators": [100, 300],
            "clf__max_depth": [None, 20],
            "clf__class_weight": [None, "balanced"],
        },
    ),
}
 

In [ ]:
parser = argparse.ArgumentParser(description="Train and evaluate text classification models.")
parser.add_argument("--data", "-d", type=str, default= "data/combined.csv", help = "Input data to feed the model")
parser.add_argument("--folds", "-f", type=int, default=5, help="Number of folds for cross-validation")
args = parser.parse_args()

cv = StratifiedKFold(n_splits=args.folds, shuffle=True, random_state=42)


In [ ]:
for model, (pipeline, param_grid) in MODELS.items():
    grid = 